# Votations fédérales suisses — Récupération et exploration des données

**Objectif de ce notebook :** récupérer l'intégralité du tableau historique de
l'OFS (`px-x-1703030000_101`, votations communales depuis 1960) via son API,
puis explorer sa couverture et sa qualité, avant tout traitement.

Le nettoyage et la mise en forme (exclusion des catégories hors périmètre,
passage au format large) sont faits dans un notebook séparé, à partir du
fichier brut produit ici (`data/raw/`), avec sortie dans `data/processed/`.

## 1. Imports

In [2]:
import os
import time

import requests
import pandas as pd
from pyjstat import pyjstat

## 2. Récupération des données (API OFS / PX-Web)

L'interface web STAT-TAB refuse les sélections de plus de 2 500 000 cellules.
On interroge donc directement l'API PX-Web qui l'alimente, en découpant la
requête en lots d'objets de votation.

Deux contraintes rencontrées en pratique, et comment on les gère :
- le serveur exige un **cookie de session** obtenu en visitant d'abord la page
  HTML normale (`session.get(PAGE_URL)`) avant d'appeler l'API ;
- le serveur **bloque les requêtes POST trop volumineuses** (indépendamment de
  la limite de 2,5M cellules) — d'où des lots de 5 objets à la fois plutôt
  qu'un plus grand nombre.

La cellule de téléchargement est **longue (~5-10 min, ~103 appels)** : le
fichier brut est donc sauvegardé dans `data/raw/`, et rechargé directement aux
exécutions suivantes du notebook plutôt que re-téléchargé.

### 2.1 Configuration et session

In [15]:
def find_project_root(marker=".git"):
    path = os.getcwd()
    while True:
        if os.path.exists(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:  # on a atteint la racine du disque sans trouver le marqueur
            raise FileNotFoundError(f"Impossible de trouver '{marker}' en remontant depuis {os.getcwd()}")
        path = parent

PROJECT_ROOT = find_project_root()
RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)
RAW_CSV = os.path.join(RAW_DIR, "votations_communes_1960_2026_brut.csv")

print("Racine du projet :", PROJECT_ROOT)
print("CSV attendu ici  :", RAW_CSV)

URL = "https://www.pxweb.bfs.admin.ch/api/v1/fr/px-x-1703030000_101/px-x-1703030000_101.px"
PAGE_URL = "https://www.pxweb.bfs.admin.ch/pxweb/fr/px-x-1703030000_101/px-x-1703030000_101/px-x-1703030000_101.px/"

HEADERS_GET = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "fr-CH,fr;q=0.9",
    "Referer": PAGE_URL,
}

HEADERS_POST = {
    **HEADERS_GET,
    "Content-Type": "application/json;charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
}

session = requests.Session()
session.headers.update(HEADERS_GET)
session.get(PAGE_URL)

Racine du projet : C:\Users\Victus\Documents\heig_vd\gml\GML-Swiss-Voting
CSV attendu ici  : C:\Users\Victus\Documents\heig_vd\gml\GML-Swiss-Voting\data\raw\votations_communes_1960_2026_brut.csv


<Response [200]>

### 2.2 Téléchargement (ou chargement du CSV déjà récupéré)

In [ ]:
EXPECTED_MIN_ROWS = 8_000_000  # seuil approximatif pour détecter un fichier tronqué

if os.path.exists(RAW_CSV) and pd.read_csv(RAW_CSV, usecols=[0]).shape[0] >= EXPECTED_MIN_ROWS:
    print(f"Fichier brut déjà présent et complet ({RAW_CSV}), chargement direct (pas de nouvel appel API).")
    df_raw = pd.read_csv(RAW_CSV)
else:
    if os.path.exists(RAW_CSV):
        print("Fichier existant mais incomplet, nouveau téléchargement...")
    print("Récupération de la métadonnée du tableau...")
    meta = session.get(URL).json()
    objets = next(v for v in meta["variables"] if "objet" in v["text"].lower())["values"]
    print(f"{len(objets)} objets de votation trouvés.")

    BATCH_SIZE = 5
    n_batches = len(objets) // BATCH_SIZE + 1
    dfs = []
    for i in range(0, len(objets), BATCH_SIZE):
        print(f"Lot {i // BATCH_SIZE + 1}/{n_batches}...")
        query = {
            "query": [
                {
                    "code": v["code"],
                    "selection": {
                        "filter": "item",
                        "values": objets[i:i + BATCH_SIZE] if "objet" in v["text"].lower() else v["values"],
                    },
                }
                for v in meta["variables"]
            ],
            "response": {"format": "json-stat2"},
        }
        r = session.post(URL, json=query, headers=HEADERS_POST)
        if r.status_code != 200:
            print(f"  Erreur {r.status_code} sur ce lot: {r.text[:300]}")
            continue
        dfs.append(pyjstat.Dataset.read(r.text).write("dataframe"))
        time.sleep(2)

    print(f"{len(dfs)} lots récupérés avec succès sur {n_batches}")
    df_raw = pd.concat(dfs, ignore_index=True)
    df_raw.to_csv(RAW_CSV, index=False, encoding="utf-8-sig")
    print(f"Sauvegardé dans {RAW_CSV}")

df_raw.shape

Récupération de la métadonnée du tableau...
511 objets de votation trouvés.
Lot 1/103...
Lot 2/103...
Lot 3/103...
Lot 4/103...
Lot 5/103...
Lot 6/103...
Lot 7/103...
Lot 8/103...
Lot 9/103...
Lot 10/103...
Lot 11/103...
Lot 12/103...
Lot 13/103...
Lot 14/103...
Lot 15/103...
Lot 16/103...
Lot 17/103...
Lot 18/103...
Lot 19/103...
Lot 20/103...
Lot 21/103...
Lot 22/103...
Lot 23/103...
Lot 24/103...
Lot 25/103...
Lot 26/103...
Lot 27/103...
Lot 28/103...
Lot 29/103...
Lot 30/103...
Lot 31/103...
Lot 32/103...
Lot 33/103...
Lot 34/103...
Lot 35/103...
Lot 36/103...
Lot 37/103...
Lot 38/103...
Lot 39/103...
Lot 40/103...
Lot 41/103...
Lot 42/103...
Lot 43/103...
Lot 44/103...
Lot 45/103...
Lot 46/103...
Lot 47/103...
Lot 48/103...
Lot 49/103...
Lot 50/103...
Lot 51/103...
Lot 52/103...
Lot 53/103...
Lot 54/103...
Lot 55/103...
Lot 56/103...
Lot 57/103...
Lot 58/103...
Lot 59/103...
Lot 60/103...
Lot 61/103...
Lot 62/103...
Lot 63/103...
Lot 64/103...
Lot 65/103...
Lot 66/103...
Lot 67/10

## 3. Vue d'ensemble des données brutes

Format **long** (une ligne par commune/canton/district × objet de votation ×
indicateur), tel que renvoyé par l'API. On identifie les noms de colonnes
exacts une seule fois, pour les réutiliser dans tout le reste du notebook.

In [ ]:
col_geo = [c for c in df_raw.columns if "Kanton" in c or "Commune" in c][0]
col_objet = [c for c in df_raw.columns if "Datum" in c or "objet" in c.lower()][0]
col_resultat = "Résultat"
col_valeur = "value"

print("Colonne géographique :", col_geo)
print("Colonne date/objet   :", col_objet)
print("Colonne indicateur   :", col_resultat)
print("Colonne valeur       :", col_valeur)

df_raw.head()

### Structure hiérarchique des données

Pour **chaque objet voté**, le tableau contient les résultats à 4 niveaux
géographiques emboîtés, dans cet ordre :
```
Suisse
└─ Canton
└─ District
└─ Commune
```

Ce niveau est encodé dans le **texte** de la colonne géographique par un préfixe :

| Préfixe | Niveau |
|---|---|
| *(aucun)* | Suisse |
| `-` | Canton |
| `>>` | District |
| `......` | Commune |

Pour une unité géographique donnée et un objet donné, il y a **7 lignes** (une
par indicateur : Electeurs inscrits, Bulletins rentrés, Participation en %,
Bulletins valables, Oui, Non, Oui en %), d'où un bloc de 7 lignes qui se
répète à chaque niveau, pour chaque objet, pour chaque unité géographique.

Deux particularités à anticiper pour la suite :
- certains petits cantons (Uri, Zoug, Glaris...) n'ont pas de vrai niveau
  district : leurs communes sont rattachées directement au canton ;
- chaque canton a aussi des entités techniques au même niveau que les
  districts (Suisses de l'étranger, votes non attribués...), traitées en
  section 5 (nettoyage), pas de vraies communes géographiques.

Cette hiérarchie Suisse → canton → district → commune, présente dans un seul
tableau, est ce qui permettra l'analyse « à contre-courant » du cahier des
charges (comparer une commune au reste de son canton, obtenu en soustrayant
ses voix du total cantonal).

## 4. Exploration et contrôle qualité

### 4.1 Couverture temporelle et géographique

In [ ]:
n_objets = df_raw[col_objet].nunique()
n_geo = df_raw[col_geo].nunique()
dates = sorted(df_raw[col_objet].str.extract(r"(\d{4}-\d{2}-\d{2})")[0].dropna().unique())

print(f"{n_objets} objets de votation distincts")
print(f"{n_geo} unités géographiques distinctes (Suisse, cantons, districts, communes)")
print(f"{len(dates)} dates de votation, de {dates[0]} à {dates[-1]}")

### 4.2 Valeurs manquantes

Deux causes distinctes, documentées dans les métadonnées du tableau OFS et
confirmées empiriquement ci-dessous :

1. **1960-1980** : couverture incomplète pour les cantons FR, AG et TI
   (annoncé explicitement par l'OFS) — gros volume de manquants concentré sur
   cette période.
2. **Après 1981** : manquants résiduels constants (même nombre d'une année à
   l'autre), qui ne sont **pas de vraies données absentes** mais des
   catégories géographiques techniques du référentiel sans valeur à
   renseigner pour la plupart des votations — Suisses de l'étranger
   (`*-Ausland-CH`), votes non attribués (`*-Anderes`), vote par
   correspondance compté à part pour certains cantons
   (`*-Korrespondenzweg`), communes sans bureau de vote propre
   (« Urne commune »).

In [ ]:
manquants = df_raw[df_raw[col_valeur].isna()]
par_annee = manquants[col_objet].str.extract(r"(\d{4})")[0].value_counts().sort_index()

print("Valeurs manquantes par année (extrait) :")
print(par_annee.head(10))
print("...")
print(par_annee.tail(10))

In [ ]:
post_1981 = manquants[manquants[col_objet].str.extract(r"(\d{4})")[0].astype(int) >= 1981]
post_1981[col_geo].value_counts().head(15)

### 4.3 Doublons

In [ ]:
doublons = df_raw.duplicated(subset=[col_geo, col_objet, col_resultat], keep=False)
print(f"{doublons.sum()} lignes dupliquées sur {len(df_raw)} (clé : géo × objet × indicateur)")

In [ ]:
print("Dossier de travail actuel :", os.getcwd())
print("Chemin absolu du CSV      :", os.path.abspath(RAW_CSV))